# 発展のノート：点の先へ

本編の実習ノートのあとに、もっと見たい人のためのノートです。本編と同じMediaPipeを使い、2つのことを試します（手指は本編の実習④に移しました）。

- A：肩の挙上面と挙上角（屈曲と外転を分ける）
- B：代償動作（肘の曲がり・体幹の傾き）

## このノートの使い方（本編と同じ）

- 上のセルから順に、全部押してください。下のセルは、上のセルで作ったものを使います。
- このノートだけで動きます（本編のノートを先に押しておく必要はありません）。
- 中身を読み飛ばしてよいセルには、そう書いてあります。

## 使うデータ

- 肩：REHAB24-6（Masaryk大学、CC BY-NC 4.0）の9番の人。正しく行った5回（正面・斜め）と、誤って行った5回（正面）。正解はモーションキャプチャー。


## 1. 道具を読み込む

本編と同じです。1〜2分かかります。

In [ ]:
!pip install -q mediapipe==0.10.35 matplotlib-fontja
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja
from scipy.signal import find_peaks
print('道具の準備ができました')

## 2. データとモデルを持ってくる

In [ ]:
URL = 'https://k0heiun0.github.io/pt_seminar/data/'
!wget -q {URL}rehab24/front.mp4 {URL}rehab24/oblique.mp4 {URL}rehab24/front_incorrect.mp4
!wget -q {URL}rehab24/front_truth.csv {URL}rehab24/oblique_truth.csv
!wget -q {URL}rehab24/front_incorrect_truth.csv
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
print('データを持ってきました')

## 3. 道具の関数を作る（中身は読み飛ばしてよい）

| 関数 | 役割 | 入力 | 出力 |
|---|---|---|---|
| `track_pose` | 全コマで体の33点を出す（本編と同じ） | 動画の名前 | 2Dの点、3Dの点、1秒のコマ数 |
| `angle_between_vectors` | 2本の矢印のなす角（本編と同じ） | 2本の矢印 | 角度 |
| `load_truth` | 正解のファイルの1列を読む | ファイル名、列の番号 | 正解の値 |

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision


def _frames(file):
    video = cv2.VideoCapture(file)
    fps = video.get(cv2.CAP_PROP_FPS)
    k = 0
    while True:
        ok, frame = video.read()
        if not ok:
            break
        yield k, fps, frame
        k += 1


def track_pose(file):
    options = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='pose_landmarker_full.task'),
        running_mode=vision.RunningMode.VIDEO, num_poses=1)
    points2d, points3d = [], []
    with vision.PoseLandmarker.create_from_options(options) as model:
        for k, fps, frame in _frames(file):
            h, w = frame.shape[:2]
            image = mp.Image(image_format=mp.ImageFormat.SRGB,
                             data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = model.detect_for_video(image, int(k * 1000 / fps))
            if result.pose_landmarks:
                points2d.append([[p.x * w, p.y * h] for p in result.pose_landmarks[0]])
                points3d.append([[p.x, p.y, p.z] for p in result.pose_world_landmarks[0]])
            else:
                points2d.append(np.full((33, 2), np.nan))
                points3d.append(np.full((33, 3), np.nan))
    return np.array(points2d), np.array(points3d), fps


def angle_between_vectors(v1, v2):
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    cos_angle = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))


def load_truth(file, column):
    return np.genfromtxt(file, delimiter=',', skip_header=1, usecols=column)

---
# A　肩の挙上面と挙上角

本編の肩挙上角は「どこまで上げたか」だけで、屈曲と外転を区別しませんでした。体幹（胸郭）に座標の軸を立てると、上腕の向きを2つに分けられます。

- 挙上角：どこまで上げたか（本編と同じ）
- 挙上面：どちら向きに上げたか（0°＝真横＝外転、+90°＝真前＝屈曲）

国際バイオメカニクス学会（ISB）の肩の角度の考え方です。

## A1. 胸郭の座標の軸を立てる関数

3Dの点から、3本の軸を作ります。上（両股の中点→両肩の中点）、右（左肩→右肩）、前（上と右に直角な向き。つま先の向きで前後をそろえる）。上腕の向きを、この3本の軸で読みます。

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)


def plane_and_elevation(p):
    # 1. 胸郭の3本の軸を作る
    up = unit((p[11] + p[12]) / 2 - (p[23] + p[24]) / 2)
    right = p[12] - p[11]
    right = unit(right - np.dot(right, up) * up)
    front = np.cross(up, right)
    if np.dot(front, p[32] - p[30]) < 0:      # つま先の向きを前にそろえる
        front = -front
    # 2. 上腕の向きを3本の軸で読む
    arm = unit(p[14] - p[12])
    elevation = angle_between_vectors(arm, -up)
    plane = np.degrees(np.arctan2(np.dot(arm, front), np.dot(arm, right)))
    return elevation, plane

## A2. 正面の動画で、正解と比べる

挙上面は、腕を下ろしているときには意味がないので、正解の挙上角が30°を超えたコマだけで比べます（正解のファイルでは、そのほかのコマは空欄）。

In [ ]:
def compare_plane(name):
    p2, p3, fps = track_pose(name + '.mp4')
    result = np.array([plane_and_elevation(p) for p in p3])
    true_elev = load_truth(name + '_truth.csv', 2)
    true_plane = load_truth(name + '_truth.csv', 3)
    up = ~np.isnan(true_plane)
    elev_err = np.nanmean(np.abs(result[:, 0] - true_elev))
    plane_err = np.nanmean(np.abs(result[up, 1] - true_plane[up]))
    print(name, '挙上角の差', round(elev_err, 1), '挙上面の差', round(plane_err, 1),
          '挙上面（正解）', round(np.nanmedian(true_plane), 1),
          '挙上面（MediaPipe）', round(np.nanmedian(result[up, 1]), 1))
    return result, true_plane, fps

front_result, front_true_plane, fps = compare_plane('front')

## A3. 斜めの動画でも比べ、グラフにする

In [ ]:
oblique_result, oblique_true_plane, fps = compare_plane('oblique')

for name, result, true_plane in [('正面', front_result, front_true_plane),
                                 ('斜め', oblique_result, oblique_true_plane)]:
    t = np.arange(len(result)) / fps
    shown = np.where(np.isnan(true_plane), np.nan, result[:, 1])
    plt.plot(t, true_plane, color='black', label=name + '：正解')
    plt.plot(t, shown, label=name + '：MediaPipe')
plt.axhline(90, color='gray', linestyle=':')
plt.axhline(0, color='gray', linestyle=':')
plt.xlabel('時間 [秒]')
plt.ylabel('挙上面 [度]（0＝真横、90＝真前）')
plt.legend()
plt.show()

## A4. Aの問い

1. 挙上面は、正解からどれくらいずれたか。屈曲（90°）と外転（0°）を見分けるには足りるか。
2. 外転（0°）と肩甲骨面での挙上（約30°）を見分けるには足りるか。
3. 正面から撮ると、挙上面はどちらに偏ったか。それはなぜか（挙上面を決める前後の成分は、正面から見ると奥行きになる）。
4. 肩の回旋（内旋・外旋）は、なぜ単眼の動画では難しいのか。

---
# B　代償動作を見つける

REHAB24-6の9番の人は、理学療法士の指示で「誤った」外転も5回行っています。正しく行った回と誤った回で、肘の曲がりと体幹の傾きを比べます。

## B1. 肘の屈曲角と体幹の傾きの関数

- 肘の屈曲角：肘を軸に、肘→肩と肘→手首のなす角を180°から引く（伸ばすと0°）
- 体幹の傾き：最初の10コマの体幹の向きから、何度傾いたか

In [ ]:
def elbow_flexion(p):
    return 180 - angle_between_vectors(p[12] - p[14], p[16] - p[14])


def trunk_lean(points):
    trunk = (points[:, 11] + points[:, 12]) / 2 - (points[:, 23] + points[:, 24]) / 2
    base = np.nanmean(trunk[:10], axis=0)
    return np.array([angle_between_vectors(t, base) for t in trunk])

## B2. 正しい回と誤った回で比べる

挙上のピークごとに、その瞬間の肘の屈曲角と体幹の傾きを並べます。正解（モーションキャプチャー）の値も並べます。

In [ ]:
def at_peaks(name):
    p2, p3, fps = track_pose(name + '.mp4')
    elev = np.array([angle_between_vectors(p[14] - p[12],
                     (p[23] + p[24]) / 2 - (p[11] + p[12]) / 2) for p in p3])
    elbow = np.array([elbow_flexion(p) for p in p3])
    lean = trunk_lean(p3)
    peaks, _ = find_peaks(np.nan_to_num(elev), height=60, prominence=30)
    true_elbow = load_truth(name + '_truth.csv', 4)
    true_lean = load_truth(name + '_truth.csv', 5)
    print(name, 'ピークの数', len(peaks))
    print('  肘の屈曲角  MediaPipe', np.round(elbow[peaks], 0), ' 正解', np.round(true_elbow[peaks], 0))
    print('  体幹の傾き  MediaPipe', np.round(lean[peaks], 1), ' 正解', np.round(true_lean[peaks], 1))
    return elbow, true_elbow, fps

ok_elbow, ok_true, fps = at_peaks('front')
ng_elbow, ng_true, fps = at_peaks('front_incorrect')

## B3. 肘の屈曲角をグラフにする

In [ ]:
for name, elbow, true in [('正しい回', ok_elbow, ok_true), ('誤った回', ng_elbow, ng_true)]:
    t = np.arange(len(elbow)) / fps
    plt.plot(t, true, color='black', linewidth=1)
    plt.plot(t, elbow, label=name + '（MediaPipe）')
plt.xlabel('時間 [秒]')
plt.ylabel('肘の屈曲角 [度]（黒は正解）')
plt.legend()
plt.show()

## B4. Bの問い

1. 誤った回の「誤り」は、何だったと読めるか（肘・体幹・挙上角のどれが違うか）。
2. MediaPipeは、その違いを正解と同じ向きにとらえられたか。
3. 臨床で代償を数で記録するなら、どの撮り方・どの指標を使うか。

---
# C　いろいろなモデルがある

MediaPipeは軽く、その場で動かせるモデルです。もっと大きなモデルもあります。

| モデル | 特徴 | 動かすには |
|---|---|---|
| SAM 3D Body / SAM-Body4D（Meta、2025年） | 体の表面（メッシュ）と手まで推定する。ベンチマークで最上位 | ColabのGPUと、Hugging Faceの利用申請が必要。推論用のノート`00_body4d_inference.ipynb` |
| GEM-X（NVIDIA、2026年） | 体・手・顔の77点。動くカメラにも対応 | GPUが必要 |
| PromptHMR、GVHMR | 動画向けの人体モデル。研究でよく使われる | GPUと、SMPLの利用登録が必要 |

大きなモデルが、リハビリの角度でも正確とは限りません。本編と同じように、正解と比べて確かめてから使います。

# D　研究の出口：筋骨格モデルへ

臨床研究で、肩の3つの角度（挙上面・挙上角・回旋）を厳密に出したいときは、点を筋骨格モデル（OpenSim）に当てはめる方法があります（逆運動学）。

- OpenCap（スタンフォード大学）：スマホの動画から、OpenSimの関節角を出す。クラウドで無料
- Sports2D・Pose2Sim：動画から、OpenSimの逆運動学までを1つの流れで行う道具
- 単眼カメラでは、肩の回旋や挙上面の誤差がまだ大きい（挙上角8.5°、挙上面15.9°、回旋21.2°という報告がある）